# Random Forest cho tiểu mục 5.3
Nguyễn Ngọc Tuấn Anh — Nhóm 8

Notebook hiển thị kết quả đã chạy từ CSV. Đổi `RUN_EXPERIMENT = True` ở ô đầu để tự chạy lại đủ 24 lượt validation và lượt fit cuối. Chạy lần lượt các ô từ trên xuống. Code dùng chung ở `common/` là bản đã gửi của TV2.

In [ ]:
from pathlib import Path
import json, pandas as pd
from IPython.display import display, Image
import sys
repo_root = Path.cwd().resolve()
if not (repo_root / "run_rf.py").exists():
    repo_root = repo_root.parent
if not (repo_root / "run_rf.py").exists():
    raise FileNotFoundError("Mở notebook từ repo hoặc thư mục notebooks của repo")
sys.path.insert(0, str(repo_root))
import run_rf as rf
RUN_EXPERIMENT = False  # True: chạy lại mô hình; False: đọc kết quả thực tế đã lưu
ROOT = rf.ROOT
print("Thư mục:", ROOT)
print("Chế độ:", "Chạy lại thực nghiệm" if RUN_EXPERIMENT else "Xem kết quả đã chạy và lưu")

## Bước 1 Kiểm tra dữ liệu
Minh chứng A: train/test và số cặp học ở các cửa sổ.

In [2]:
if RUN_EXPERIMENT:
    rf.prepare()
else:
    rf.load_inputs()
    rf.check_protocol()
display(pd.read_csv(ROOT / "results/data_checks_rf.csv")[["label","train_end","val_start","val_end","n_train_months","n_supervised_rows"]])
protocol = json.loads((ROOT / "protocol_rf.json").read_text(encoding="utf-8"))
print("Hash khớp protocol gốc:", protocol["original_hash_matches"])
print("Hash CSV thực:", protocol["data_sha256"])

Hash khớp protocol gốc: False
Hash CSV thực: 0cfff966678a4f20a0a48a87f090cfcc7eee1b3029010c762fa66e93a42f7315


,label,train_end,val_start,val_end,n_train_months,n_supervised_rows
0,Fold 1,2020-11-30,2020-12-31,2021-11-30,47,40
1,Fold 2,2021-05-31,2021-06-30,2022-05-31,53,46
2,Fold 3,2021-11-30,2021-12-31,2022-11-30,59,52
3,Full train,2022-11-30,2022-12-31,2023-11-30,71,64


## Bước 2 Chạy validation và khóa cấu hình
Minh chứng B: tám cấu hình, ba RMSE, RMSE trung bình và cấu hình chọn. Không dùng test để chọn.

In [3]:
if RUN_EXPERIMENT:
    rf.run_cv()
cv = pd.read_csv(ROOT / "results/cv_rf_summary.csv").sort_values("config_id")
cols = ["config_id","max_depth","min_samples_leaf","max_features","rmse_fold1","rmse_fold2","rmse_fold3","mean_cv_rmse","selected"]
display(cv[cols].round(2))
best = json.loads((ROOT / "results/best_rf.json").read_text(encoding="utf-8"))
print("Cấu hình đã khóa:", best["config"])
print("Thời điểm khóa trước test:", best["locked_at_utc"])
print("Mean CV RMSE:", best["mean_cv_rmse"])

Cấu hình đã khóa: {'config_id': 'RF05', 'n_estimators': 300, 'max_depth': 3, 'min_samples_leaf': 3, 'max_features': 0.7, 'random_state': 42}
Thời điểm khóa trước test: 2026-11-12T19:32:21.337830+00:00
Mean CV RMSE: 814.1415081730869


,config_id,max_depth,min_samples_leaf,max_features,rmse_fold1,rmse_fold2,rmse_fold3,mean_cv_rmse,selected
2,RF01,2,3,0.7,337.54,805.59,1536.77,893.30,False
3,RF02,2,3,1.0,473.32,774.71,1458.41,902.15,False
6,RF03,2,5,0.7,754.62,823.45,1491.19,1023.09,False
4,RF04,2,5,1.0,446.02,883.98,1535.19,955.06,False
0,RF05,3,3,0.7,328.61,732.80,1381.02,814.14,True
1,RF06,3,3,1.0,450.71,762.05,1377.54,863.43,False
7,RF07,3,5,0.7,770.07,838.76,1576.05,1061.63,False
5,RF08,3,5,1.0,479.58,902.84,1660.64,1014.35,False


## Bước 3 Fit cuối và dự báo 12 bước
Minh chứng C: 12 actual/pred, điểm RF, so sánh ARIMA và kiểm tra tích hợp. Bias dương là dự báo thiếu.

In [4]:
if RUN_EXPERIMENT:
    rf.run_forecast()
forecast = pd.read_csv(ROOT / "results/forecast_rf.csv")
display(forecast[["date","horizon","actual","pred"]].round(2))
display(pd.read_csv(ROOT / "results/metrics_rf.csv"))
display(pd.read_csv(ROOT / "results/comparison_rf_arima.csv"))
display(pd.read_csv(ROOT / "results/integration_checks_rf.csv"))

,date,horizon,actual,pred
0,2022-12-31,1,115665.0,115316.96
1,2023-01-31,2,119377.0,118294.06
2,2023-02-28,3,122322.0,121510.15
3,2023-03-31,4,125478.0,124269.32
4,2023-04-30,5,129728.0,127443.45
5,2023-05-31,6,133975.0,130865.18
6,2023-06-30,7,138353.0,133904.79
7,2023-07-31,8,142537.0,137388.73
8,2023-08-31,9,148730.0,141090.65
9,2023-09-30,10,152690.0,144364.55


,rmse,mae,smape,bias,model,config_id,mean_cv_rmse,n_train,n_supervised,seed,runtime_s
0,5243.320964,4242.907515,2.959396,4242.907515,Random Forest,RF05,814.141508,71,64,42,0.537198


,model,rmse,mae,smape,bias,rmse_improvement_vs_arima_pct
0,"ARIMA(2,2,2) supplied forecast",2723.513206,2090.419366,1.444409,1260.065761,0.00000
1,Random Forest,5243.320964,4242.907515,2.959396,4242.907515,-92.52049


,check,status
0,full_train_64_rows,PASS
1,label_scaler_train_only,PASS
2,recursive_lag1_uses_previous_prediction,PASS
3,features_exclude_current_target,PASS
4,future_actual_mutation_does_not_change_predict...,PASS
5,cv_288_predictions,PASS
6,growth_inversion,PASS
7,negative_growth_not_clipped,PASS


## Bước 4 Hình minh chứng
Chèn ảnh PNG trực tiếp vào báo cáo. Khi chụp bằng chứng chạy code, giữ cả ô code và phần output liên quan.

In [5]:
if RUN_EXPERIMENT:
    rf.make_figures()
for name in ["rf_validation_table.png", "rf_forecast.png", "rf_errors.png"]:
    display(Image(filename=str(ROOT / "figures" / name), width=950))

## Kết luận từ lần chạy đã lưu
RF05 có mean CV RMSE 814,14 xe; test RMSE 5.243,32 xe, MAE 4.242,91 xe và sMAPE 2,96%. RF dự báo thấp cả 12 tháng. ARIMA từ CSV gửi đạt RMSE 2.723,51 xe, thấp hơn RF. Không chỉnh tham số theo test để đổi kết luận.

Hash CSV không khớp protocol gốc, cần nhóm xác nhận nguồn khi tích hợp; lần chạy này đã ghi hash của chính CSV được gửi. Word là bản báo cáo của lần chạy được giao; nếu dữ liệu hoặc thiết kế thay đổi, cần cập nhật báo cáo theo kết quả mới.

### Hình đã xuất từ lần chạy

![Dự báo RF](../figures/rf_forecast.png)

![Validation RF](../figures/rf_validation_table.png)